# Cleaning

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn pydtmc


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [38]:
import pandas as pd
df=pd.read_csv('../data/cville_weather_full.csv')

/var/folders/b1/znq25z59401dlz_b0y0j_2c00000gn/T/ipykernel_89322/1184279021.py:2: DtypeWarning: Columns (0: DAPR_ATTRIBUTES, 1: DASF_ATTRIBUTES, 2: MDPR_ATTRIBUTES, 3: MDSF_ATTRIBUTES, 4: WESD_ATTRIBUTES, 5: WT01_ATTRIBUTES, 6: WT03_ATTRIBUTES, 7: WT05_ATTRIBUTES, 8: WT06_ATTRIBUTES, 9: WT08_ATTRIBUTES, 10: WT11_ATTRIBUTES, 11: WT14_ATTRIBUTES, 12: WT16_ATTRIBUTES, 13: WT18_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('../data/cville_weather_full.csv')


In [39]:
df.columns
df.head()

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,WT08,WT08_ATTRIBUTES,WT11,WT11_ATTRIBUTES,WT14,WT14_ATTRIBUTES,WT16,WT16_ATTRIBUTES,WT18,WT18_ATTRIBUTES
0,USC00441593,1893-01-01,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",399.0,",,6,",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,",,6"
1,USC00441593,1893-01-02,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USC00441593,1893-01-03,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USC00441593,1893-01-04,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",66.0,",,6,",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USC00441593,1893-01-05,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
cols_to_keep = ["STATION", "DATE", "NAME", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "WT01", "WT03", "WT04", 
                "WT05", "WT06", "WT08", "WT11", "WT14", "WT16", "WT18"]

df = df[cols_to_keep]

In [41]:
df["DATE"] = pd.to_datetime(df["DATE"], format="mixed", errors="coerce")
df = df.dropna(subset=["DATE"])

df = df[(df["DATE"] >= "2000-01-01") & (df["DATE"] <= "2025-12-31")]

df = df.sort_values("DATE")

In [42]:
print(df["DATE"].min())
print(df["DATE"].max())
print(len(df))

2000-01-01 00:00:00
2025-12-31 00:00:00
9437


In [43]:
df["TMAX"] = df["TMAX"] / 10.0
df["TMIN"] = df["TMIN"] / 10.0

In [44]:
df["TMAX"] = (df["TMAX"] * 1.8) + 32
df["TMIN"] = (df["TMIN"] * 1.8) + 32

In [45]:
df["TMAX"] = df["TMAX"].round(1)
df["TMIN"] = df["TMIN"].round(1)

In [46]:
for col in ["TMAX", "TMIN", "PRCP"]:
    df[col] = df[col].fillna(df[col].median())

In [47]:
for col in ["SNOW", "SNWD"]:
    df[col] = df[col].fillna(0)

In [48]:
wt_cols = ["WT01","WT03","WT04","WT05","WT06","WT08","WT11","WT14","WT16","WT18"]

df[wt_cols] = df[wt_cols].replace(r'^\s*$', None, regex=True)
df[wt_cols] = df[wt_cols].apply(pd.to_numeric, errors='coerce')

df[wt_cols] = df[wt_cols].fillna(0).astype(int)

In [49]:
df["PRCP"].describe()

count    9437.000000
mean       32.897001
std        93.433433
min         0.000000
25%         0.000000
50%         0.000000
75%        13.000000
max      1765.000000
Name: PRCP, dtype: float64

In [50]:
df["TMAX"].describe()

count    9437.000000
mean       67.744643
std        17.616205
min        17.100000
25%        54.000000
50%        70.000000
75%        82.900000
max       105.100000
Name: TMAX, dtype: float64

In [51]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["season"] = df["DATE"].dt.month.apply(get_season)

In [52]:
df.to_csv("../data/cville_weather_cleaned.csv", index=False)